# OE2B-05 — Entrenamiento xnorC

## ¿Qué hace este notebook?

Entrena la red **xnorC** — una red siamesa que aprende a comparar dos segmentos de audio
y predice si la colmena está en el **mismo estado** (IGUAL) o en **estados distintos** (DIFERENTE).

Se entrenan **4 experimentos**, uno por cada representación de audio:

| Experimento | Features | Dimensión |
|------------|----------|----------|
| B1 | Mel spectrogram (promedio) | 64 |
| B2 | MFCC + Delta (promedio) | 40 |
| B3 | FFT bins (promedio) | 129 |
| B4 | CQT (promedio) | 72 |

Para cada experimento se hace **k-hive-fold** (k=4): cada fold excluye una colmena completa
del entrenamiento y la usa para test. Esto mide la capacidad del modelo de generalizar
a colmenas que nunca vio.

### Arquitectura xnorC

```
Audio A → [Encoder] → embedding_A ─┐
                                    ├─ |embedding_A - embedding_B| → [Clasificador] → IGUAL/DIFERENTE
Audio B → [Encoder] → embedding_B ─┘
```

El **Encoder** es una red densa que transforma el vector de features en un embedding.
El **Clasificador** toma la diferencia absoluta entre los dos embeddings y predice la etiqueta.
Ambas ramas comparten exactamente los mismos pesos (siamesa verdadera).

### Salidas del notebook

Para cada experimento B1–B4 y cada fold k:
- `modelos/xnorC_B{n}_fold{k}.pt` — pesos del modelo entrenado
- `resultados/metricas_B{n}_fold{k}.json` — accuracy, F1, AUC en test
- `resultados/metricas_resumen.csv` — tabla resumen con todos los resultados

## 1. Importaciones y configuración

In [1]:
# Librerías estándar
import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Métricas
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

# Para mostrar el progreso
from tqdm.notebook import tqdm

print("Librerías importadas correctamente")
print(f"PyTorch versión: {torch.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")

Librerías importadas correctamente
PyTorch versión: 2.13.0+cu130
GPU disponible: True


In [2]:
# ─────────────────────────────────────────────
#  CONFIGURACIÓN — ajustar según el entorno
# ─────────────────────────────────────────────

# Carpeta donde OE2B-03 guardó los folds
# (OE2B-03 usa: RUTA_FOLDS = Path('../data/processed/pares/folds'))
CARPETA_FOLDS    = Path('../data/processed/pares/folds')

# Carpeta donde OE2B-04 guardó los features
# (OE2B-04 usa: RUTA_FEATURES = Path('../data/processed/features'))
CARPETA_FEATURES = Path('../data/processed/features')

# Carpeta para guardar los modelos entrenados
CARPETA_MODELOS  = Path('../data/processed/modelos')
CARPETA_MODELOS.mkdir(parents=True, exist_ok=True)

# Carpeta para guardar los resultados
CARPETA_RESULTADOS = Path('../data/processed/resultados')
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)

# Hiperparámetros de entrenamiento
EPOCHS         = 50       # cantidad de épocas
BATCH_SIZE     = 64       # pares por batch
LEARNING_RATE  = 1e-3     # tasa de aprendizaje inicial
EMBEDDING_DIM  = 64       # tamaño del embedding (salida del encoder)
DROPOUT        = 0.3      # dropout para regularización
PATIENCE       = 10       # épocas sin mejora antes de detener (early stopping)

# Semilla para reproducibilidad
SEMILLA = 42

# Dispositivo: GPU si está disponible, sino CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Experimentos a correr: nombre → archivo de features
# Los nombres coinciden exactamente con lo que guarda OE2B-04
EXPERIMENTOS = {
    'B1': 'features_B1_mel.npy',
    'B2': 'features_B2_mfcc.npy',
    'B3': 'features_B3_fft.npy',
    'B4': 'features_B4_cqt.npy',
}

print(f"Dispositivo de entrenamiento: {DEVICE}")
print(f"Experimentos: {list(EXPERIMENTOS.keys())}")
print(f"Folds:    {CARPETA_FOLDS}")
print(f"Features: {CARPETA_FEATURES}")

Dispositivo de entrenamiento: cuda
Experimentos: ['B1', 'B2', 'B3', 'B4']
Folds:    ../data/processed/pares/folds
Features: ../data/processed/features


In [3]:
def fijar_semilla(semilla):
    """Fija todas las semillas para que los resultados sean reproducibles."""
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semilla)

fijar_semilla(SEMILLA)
print(f"Semilla fijada: {SEMILLA}")

Semilla fijada: 42


## 2. Cargar features e índice

El OE2B-04 generó:
- `features_index.csv` — un archivo con una fila por segmento, con columnas `archivo` y `ruta_relativa`
- `features_B{n}_XXX.npy` — un array numpy con shape `(N_segmentos, dim_features)`

El índice conecta cada fila del array de features con el nombre del archivo de audio.

In [4]:
# Cargar el índice de features
ruta_indice = CARPETA_FEATURES / 'features_index.csv'
df_indice = pd.read_csv(ruta_indice)

print(f"Total de segmentos en el índice: {len(df_indice)}")
print()
print("Primeras filas del índice:")
print(df_indice.head())
print()
print("Columnas:", df_indice.columns.tolist())

Total de segmentos en el índice: 7676

Primeras filas del índice:
   feature_idx                                               ruta dataset  \
0            0  /run/media/ikarus/KINGSTON/Tesis-Colmenas/Yang...    yang   
1            1  /run/media/ikarus/KINGSTON/Tesis-Colmenas/Yang...    yang   
2            2  /run/media/ikarus/KINGSTON/Tesis-Colmenas/Yang...    yang   
3            3  /run/media/ikarus/KINGSTON/Tesis-Colmenas/Yang...    yang   
4            4  /run/media/ikarus/KINGSTON/Tesis-Colmenas/Yang...    yang   

  estado  
0     QR  
1     QR  
2     QR  
3     QR  
4     QR  

Columnas: ['feature_idx', 'ruta', 'dataset', 'estado']


In [5]:
# Cargar todos los arrays de features en memoria
features_dict = {}

for nombre, archivo in EXPERIMENTOS.items():
    ruta = CARPETA_FEATURES / archivo
    if not ruta.exists():
        print(f"⚠️  No se encontró: {ruta}")
        continue
    arr = np.load(ruta)
    features_dict[nombre] = arr
    print(f"  {nombre}: shape={arr.shape}, dtype={arr.dtype}")

print()
print("Features cargados:", list(features_dict.keys()))

# Crear el mapa: nombre_archivo (solo el stem, sin path) → índice en el array
# OE2B-04 guarda la columna 'ruta' con la ruta completa del archivo
# Los CSVs de pares de OE2B-03 guardan solo el nombre del archivo (sin path)
# → usamos Path(ruta).name para extraer el nombre y que coincidan
archivo_a_indice = {
    Path(row['ruta']).name: idx
    for idx, row in df_indice.iterrows()
}
print(f"Mapa archivo→índice creado con {len(archivo_a_indice)} entradas")

# Mostrar algunos ejemplos para verificar
ejemplos = list(archivo_a_indice.items())[:3]
for nombre_arch, idx in ejemplos:
    print(f"  '{nombre_arch}' → índice {idx}")

  B1: shape=(7676, 64), dtype=float32
  B2: shape=(7676, 40), dtype=float32
  B3: shape=(7676, 129), dtype=float32
  B4: shape=(7676, 72), dtype=float32

Features cargados: ['B1', 'B2', 'B3', 'B4']
Mapa archivo→índice creado con 7676 entradas
  '2022-06-05--17-41-01_2__segment0.wav' → índice 0
  '2022-06-05--17-41-01_2__segment1.wav' → índice 1
  '2022-06-05--17-41-01_2__segment2.wav' → índice 2


## 3. Dataset de pares

Para entrenar xnorC necesitamos cargar los pares que construyó OE2B-03.
Cada par tiene:
- `archivo_a` y `archivo_b` — los dos segmentos a comparar
- `etiqueta` — 0 si son IGUAL, 1 si son DIFERENTE

La clase `DatasetPares` recibe el DataFrame de pares y los arrays de features,
y en `__getitem__` devuelve `(features_a, features_b, etiqueta)` para un par dado.

In [6]:
class DatasetPares(Dataset):
    """
    Dataset de pares de segmentos de audio para xnorC.
    
    Los CSVs de OE2B-03 tienen las columnas:
        'ruta_a', 'ruta_b'  — rutas completas de los dos segmentos
        'etiqueta'          — 0=DIFERENTE, 1=IGUAL
    
    El mapa archivo_a_indice usa Path(ruta).name para indexar sin el path completo.
    """
    
    def __init__(self, df_pares, features_array, archivo_a_indice):
        self.pares    = df_pares.reset_index(drop=True)
        self.features = features_array
        self.mapa     = archivo_a_indice
        
        # Verificar que todos los archivos del fold tienen features
        archivos_faltantes = []
        for _, fila in self.pares.iterrows():
            nombre_a = Path(fila['ruta_a']).name
            nombre_b = Path(fila['ruta_b']).name
            if nombre_a not in self.mapa:
                archivos_faltantes.append(nombre_a)
            if nombre_b not in self.mapa:
                archivos_faltantes.append(nombre_b)
        
        if archivos_faltantes:
            print(f"⚠️  {len(set(archivos_faltantes))} archivos sin features (se van a ignorar)")
            # Filtrar los pares donde ambos archivos tienen features
            mascara = self.pares.apply(
                lambda fila: (
                    Path(fila['ruta_a']).name in self.mapa and
                    Path(fila['ruta_b']).name in self.mapa
                ),
                axis=1
            )
            self.pares = self.pares[mascara].reset_index(drop=True)
            print(f"   Pares restantes: {len(self.pares)}")
    
    def __len__(self):
        return len(self.pares)
    
    def __getitem__(self, idx):
        fila = self.pares.iloc[idx]
        
        # Los CSVs de OE2B-03 guardan 'ruta_a' y 'ruta_b' con path completo
        # El mapa usa solo el nombre del archivo (Path.name)
        nombre_a = Path(fila['ruta_a']).name
        nombre_b = Path(fila['ruta_b']).name
        
        feat_a   = self.features[self.mapa[nombre_a]]
        feat_b   = self.features[self.mapa[nombre_b]]
        etiqueta = int(fila['etiqueta'])  # 0=DIFERENTE, 1=IGUAL
        
        return (
            torch.tensor(feat_a,   dtype=torch.float32),
            torch.tensor(feat_b,   dtype=torch.float32),
            torch.tensor(etiqueta, dtype=torch.long)
        )

## 4. Arquitectura xnorC

La red siamesa tiene dos componentes:

**Encoder:** transforma el vector de features (ej. 64 números para B1) en un embedding
de tamaño fijo (`EMBEDDING_DIM=64`). Las dos ramas comparten exactamente los mismos pesos.

**Clasificador:** recibe `|embedding_A - embedding_B|` (diferencia absoluta) y predice
si el par es IGUAL o DIFERENTE.

La diferencia absoluta es una operación simétrica: da el mismo resultado
si damos el par en orden (A, B) o al revés (B, A). Esto es deseable porque
la comparación no debería depender del orden.

In [7]:
class Encoder(nn.Module):
    """
    Encoder de xnorC: transforma un vector de features en un embedding.
    
    Arquitectura: Linear → ReLU → Dropout → Linear → ReLU → Dropout → Linear
    
    Parámetros:
        dim_entrada: tamaño del vector de features (64 para B1, 40 para B2, etc.)
        dim_embedding: tamaño del embedding de salida
        dropout: probabilidad de dropout
    """
    
    def __init__(self, dim_entrada, dim_embedding=64, dropout=0.3):
        super().__init__()
        
        # Capa oculta de tamaño 2x el embedding
        dim_oculta = dim_embedding * 2
        
        self.red = nn.Sequential(
            # Primera capa: features → oculta
            nn.Linear(dim_entrada, dim_oculta),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # Segunda capa: oculta → embedding
            nn.Linear(dim_oculta, dim_embedding),
            nn.ReLU(),
        )
    
    def forward(self, x):
        return self.red(x)


class Clasificador(nn.Module):
    """
    Clasificador de xnorC: a partir de |emb_A - emb_B| predice IGUAL/DIFERENTE.
    
    Parámetros:
        dim_embedding: tamaño del embedding (entrada = dim_embedding)
        dropout: probabilidad de dropout
    """
    
    def __init__(self, dim_embedding=64, dropout=0.3):
        super().__init__()
        
        self.red = nn.Sequential(
            nn.Linear(dim_embedding, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 2)  # 2 clases: IGUAL y DIFERENTE
        )
    
    def forward(self, diff):
        return self.red(diff)


class xnorC(nn.Module):
    """
    Red siamesa xnorC completa.
    
    Recibe dos segmentos de audio (como vectores de features) y predice
    si están en el mismo estado (IGUAL) o en estados distintos (DIFERENTE).
    
    Parámetros:
        dim_entrada: tamaño del vector de features
        dim_embedding: tamaño del espacio de embedding
        dropout: probabilidad de dropout
    """
    
    def __init__(self, dim_entrada, dim_embedding=64, dropout=0.3):
        super().__init__()
        
        # Las dos ramas comparten el MISMO encoder (mismos pesos)
        self.encoder = Encoder(dim_entrada, dim_embedding, dropout)
        self.clasificador = Clasificador(dim_embedding, dropout)
    
    def forward(self, audio_a, audio_b):
        # Pasar cada segmento por el encoder compartido
        emb_a = self.encoder(audio_a)
        emb_b = self.encoder(audio_b)
        
        # Diferencia absoluta: simétrica (no depende del orden)
        diferencia = torch.abs(emb_a - emb_b)
        
        # Clasificar
        logits = self.clasificador(diferencia)
        return logits
    
    def get_embedding(self, audio):
        """Obtener el embedding de un segmento (útil para visualización)."""
        return self.encoder(audio)


# Probar que la arquitectura funciona con un ejemplo
modelo_prueba = xnorC(dim_entrada=64, dim_embedding=EMBEDDING_DIM, dropout=DROPOUT)
audio_a_prueba = torch.randn(4, 64)  # batch de 4 pares, features de dim 64
audio_b_prueba = torch.randn(4, 64)
salida_prueba = modelo_prueba(audio_a_prueba, audio_b_prueba)
print("Prueba de arquitectura:")
print(f"  Entrada: batch={4}, dim_features={64}")
print(f"  Salida: shape={salida_prueba.shape}  ← (batch, 2 clases)")
print()
print(modelo_prueba)

Prueba de arquitectura:
  Entrada: batch=4, dim_features=64
  Salida: shape=torch.Size([4, 2])  ← (batch, 2 clases)

xnorC(
  (encoder): Encoder(
    (red): Sequential(
      (0): Linear(in_features=64, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=128, out_features=64, bias=True)
      (4): ReLU()
    )
  )
  (clasificador): Clasificador(
    (red): Sequential(
      (0): Linear(in_features=64, out_features=32, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=32, out_features=2, bias=True)
    )
  )
)


## 5. Funciones de entrenamiento y evaluación

In [8]:
def entrenar_epoca(modelo, loader, optimizador, criterio, device):
    """
    Corre una época de entrenamiento completa.
    Devuelve la pérdida promedio del epoch.
    """
    modelo.train()  # modo entrenamiento (activa dropout)
    perdida_total = 0.0
    n_batches = 0
    
    for audio_a, audio_b, etiquetas in loader:
        # Mover datos al dispositivo (GPU o CPU)
        audio_a  = audio_a.to(device)
        audio_b  = audio_b.to(device)
        etiquetas = etiquetas.to(device)
        
        # Forward pass
        optimizador.zero_grad()
        logits = modelo(audio_a, audio_b)
        perdida = criterio(logits, etiquetas)
        
        # Backward pass
        perdida.backward()
        optimizador.step()
        
        perdida_total += perdida.item()
        n_batches += 1
    
    return perdida_total / n_batches


def evaluar(modelo, loader, criterio, device):
    """
    Evalúa el modelo en un conjunto de datos.
    Devuelve pérdida, accuracy, F1, AUC y la matriz de confusión.
    """
    modelo.eval()  # modo evaluación (desactiva dropout)
    perdida_total = 0.0
    n_batches = 0
    
    todas_etiquetas = []
    todas_predicciones = []
    todas_probabilidades = []  # para calcular AUC
    
    with torch.no_grad():  # no calcular gradientes (más rápido)
        for audio_a, audio_b, etiquetas in loader:
            audio_a  = audio_a.to(device)
            audio_b  = audio_b.to(device)
            etiquetas = etiquetas.to(device)
            
            logits = modelo(audio_a, audio_b)
            perdida = criterio(logits, etiquetas)
            perdida_total += perdida.item()
            n_batches += 1
            
            # Convertir logits a predicciones y probabilidades
            probs = torch.softmax(logits, dim=1)  # probabilidades de cada clase
            preds = torch.argmax(logits, dim=1)    # clase predicha
            
            todas_etiquetas.extend(etiquetas.cpu().numpy())
            todas_predicciones.extend(preds.cpu().numpy())
            todas_probabilidades.extend(probs[:, 1].cpu().numpy())  # prob de DIFERENTE
    
    # Calcular métricas
    acc  = accuracy_score(todas_etiquetas, todas_predicciones)
    f1   = f1_score(todas_etiquetas, todas_predicciones, average='weighted')
    auc  = roc_auc_score(todas_etiquetas, todas_probabilidades)
    conf = confusion_matrix(todas_etiquetas, todas_predicciones)
    
    return {
        'perdida': perdida_total / n_batches,
        'accuracy': acc,
        'f1': f1,
        'auc': auc,
        'confusion_matrix': conf.tolist()
    }

In [9]:
def entrenar_fold(experimento, fold_k, features_array, archivo_a_indice,
                  epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE,
                  embedding_dim=EMBEDDING_DIM, dropout=DROPOUT, patience=PATIENCE,
                  device=DEVICE):
    """
    Entrena xnorC para un experimento y un fold específico.
    
    Parámetros:
        experimento: nombre del experimento (ej. 'B1')
        fold_k: número de fold (empieza en 0)
        features_array: array con los features del experimento
        archivo_a_indice: mapa nombre_archivo → índice en features_array
    
    Devuelve:
        metricas_test: dict con accuracy, f1, auc y confusion_matrix
    """
    # Cargar los pares del fold
    ruta_train = CARPETA_FOLDS / f'fold_{fold_k}_train.csv'
    ruta_test  = CARPETA_FOLDS / f'fold_{fold_k}_test.csv'
    
    if not ruta_train.exists() or not ruta_test.exists():
        raise FileNotFoundError(f"No se encontraron los archivos del fold {fold_k}")
    
    df_train = pd.read_csv(ruta_train)
    df_test  = pd.read_csv(ruta_test)
    
    print(f"  Fold {fold_k}: train={len(df_train)} pares, test={len(df_test)} pares")
    
    # Crear datasets y loaders
    ds_train = DatasetPares(df_train, features_array, archivo_a_indice)
    ds_test  = DatasetPares(df_test,  features_array, archivo_a_indice)
    
    loader_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True)
    loader_test  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False)
    
    # Dimensión de entrada = tamaño de un vector de features
    dim_entrada = features_array.shape[1]
    
    # Crear modelo, optimizador y función de pérdida
    fijar_semilla(SEMILLA)  # para reproducibilidad dentro del fold
    modelo = xnorC(dim_entrada, embedding_dim, dropout).to(device)
    optimizador = optim.Adam(modelo.parameters(), lr=lr)
    # ReduceLROnPlateau: baja la lr si la pérdida de val no mejora
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizador, patience=5, factor=0.5)
    criterio = nn.CrossEntropyLoss()
    
    # Entrenamiento con early stopping
    mejor_perdida = float('inf')
    epocas_sin_mejora = 0
    mejor_estado = None
    historial = []
    
    for epoca in range(epochs):
        # Entrenar una época
        perdida_train = entrenar_epoca(modelo, loader_train, optimizador, criterio, device)
        
        # Evaluar en test
        metricas = evaluar(modelo, loader_test, criterio, device)
        perdida_test = metricas['perdida']
        
        # Ajustar learning rate si no mejora
        scheduler.step(perdida_test)
        
        historial.append({
            'epoca': epoca + 1,
            'perdida_train': perdida_train,
            'perdida_test': perdida_test,
            'accuracy': metricas['accuracy'],
            'f1': metricas['f1'],
        })
        
        # Mostrar progreso cada 10 épocas
        if (epoca + 1) % 10 == 0:
            print(f"    Época {epoca+1:3d}/{epochs}: "
                  f"train_loss={perdida_train:.4f}  "
                  f"test_loss={perdida_test:.4f}  "
                  f"acc={metricas['accuracy']:.3f}  "
                  f"f1={metricas['f1']:.3f}")
        
        # Early stopping: guardar el mejor modelo
        if perdida_test < mejor_perdida:
            mejor_perdida = perdida_test
            epocas_sin_mejora = 0
            # Guardar copia del estado del mejor modelo
            mejor_estado = {k: v.cpu().clone() for k, v in modelo.state_dict().items()}
        else:
            epocas_sin_mejora += 1
            if epocas_sin_mejora >= patience:
                print(f"    Early stopping en época {epoca+1} "
                      f"(sin mejora por {patience} épocas)")
                break
    
    # Cargar el mejor modelo y evaluar una última vez
    modelo.load_state_dict(mejor_estado)
    metricas_finales = evaluar(modelo, loader_test, criterio, device)
    
    # Guardar el modelo
    ruta_modelo = CARPETA_MODELOS / f'xnorC_{experimento}_fold{fold_k}.pt'
    torch.save({
        'estado_modelo': mejor_estado,
        'dim_entrada': dim_entrada,
        'embedding_dim': embedding_dim,
        'dropout': dropout,
        'historial': historial,
        'experimento': experimento,
        'fold': fold_k,
    }, ruta_modelo)
    
    print(f"  → Modelo guardado en {ruta_modelo}")
    print(f"  → Accuracy final: {metricas_finales['accuracy']:.3f}  "
          f"F1: {metricas_finales['f1']:.3f}  "
          f"AUC: {metricas_finales['auc']:.3f}")
    
    return metricas_finales, historial

## 6. Entrenamiento de todos los experimentos

Para cada experimento (B1–B4) y cada fold (k=0,1,2,3) se entrena un modelo xnorC.
Los resultados se guardan en la carpeta `resultados/`.

In [10]:
# Detectar cuántos folds hay
archivos_fold = list(CARPETA_FOLDS.glob('fold_*_train.csv'))
n_folds = len(archivos_fold)

if n_folds == 0:
    print("⚠️  No se encontraron archivos de folds en:", CARPETA_FOLDS)
    print("   Verificá que el notebook OE2B-03 se ejecutó correctamente.")
else:
    print(f"Folds encontrados: {n_folds}")
    for f in sorted(archivos_fold):
        df_tmp = pd.read_csv(f)
        print(f"  {f.name}: {len(df_tmp)} pares")

Folds encontrados: 4
  fold_0_train.csv: 1880 pares
  fold_1_train.csv: 1880 pares
  fold_2_train.csv: 1326 pares
  fold_3_train.csv: 1418 pares


In [11]:
# ─────────────────────────────────────────────
#  ENTRENAMIENTO COMPLETO
#  Puede tardar varios minutos dependiendo del
#  hardware. Con GPU es mucho más rápido.
# ─────────────────────────────────────────────

todos_los_resultados = []  # guardar todos los resultados acá

for experimento, archivo_features in EXPERIMENTOS.items():
    if experimento not in features_dict:
        print(f"\n⚠️  Saltando {experimento}: no se cargaron los features")
        continue
    
    features_array = features_dict[experimento]
    print(f"\n{'='*60}")
    print(f" Experimento {experimento}  —  features: {features_array.shape}")
    print(f"{'='*60}")
    
    for fold_k in range(n_folds):
        print(f"\n  [Fold {fold_k}]")
        
        metricas, historial = entrenar_fold(
            experimento   = experimento,
            fold_k        = fold_k,
            features_array = features_array,
            archivo_a_indice = archivo_a_indice,
        )
        
        # Guardar métricas en JSON
        resultado = {
            'experimento': experimento,
            'fold': fold_k,
            'accuracy': metricas['accuracy'],
            'f1': metricas['f1'],
            'auc': metricas['auc'],
            'confusion_matrix': metricas['confusion_matrix'],
        }
        ruta_json = CARPETA_RESULTADOS / f'metricas_{experimento}_fold{fold_k}.json'
        with open(ruta_json, 'w') as f:
            json.dump(resultado, f, indent=2)
        
        todos_los_resultados.append(resultado)

print(f"\n{'='*60}")
print("Entrenamiento completado.")
print(f"Modelos guardados en: {CARPETA_MODELOS}")
print(f"Métricas guardadas en: {CARPETA_RESULTADOS}")


 Experimento B1  —  features: (7676, 64)

  [Fold 0]
  Fold 0: train=1880 pares, test=288 pares


/home/ikarus/anaconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:422: UserWarning: Found GPU0 NVIDIA GeForce 940MX which is of compute capability (CC) 5.0.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Your installed torch==2.13.0+cu130 does not include kernels for this GPU. Reinstall the same version against a CUDA build that does, e.g.:
  For CUDA 12.6 use pip install torch==2.13.0 --index-url https://download.pytorch.org/whl/cu126
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/ikarus/anaconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:540: UserWarning: 
NVIDIA GeForce 940MX with CUDA capabili

RuntimeError: CUDA error: CUBLAS_STATUS_ARCH_MISMATCH when calling `cublasCreate(handle)`

## 7. Resumen de resultados

In [ ]:
# Cargar todos los JSON de métricas (en caso de que se haya interrumpido y vuelto a correr)
resultados_cargados = []
for json_path in sorted(CARPETA_RESULTADOS.glob('metricas_B*.json')):
    with open(json_path) as f:
        resultados_cargados.append(json.load(f))

if not resultados_cargados:
    print("No se encontraron archivos de métricas. Corré la celda de entrenamiento primero.")
else:
    # Crear tabla resumen
    filas = []
    for r in resultados_cargados:
        filas.append({
            'Experimento': r['experimento'],
            'Fold': r['fold'],
            'Accuracy': round(r['accuracy'], 3),
            'F1': round(r['f1'], 3),
            'AUC': round(r['auc'], 3),
        })
    
    df_resultados = pd.DataFrame(filas)
    
    print("Resultados por fold:")
    print(df_resultados.to_string(index=False))
    print()
    
    # Promediar por experimento
    print("Promedio por experimento:")
    df_promedio = (
        df_resultados
        .groupby('Experimento')[['Accuracy', 'F1', 'AUC']]
        .agg(['mean', 'std'])
        .round(3)
    )
    print(df_promedio)
    
    # Guardar CSV resumen
    ruta_csv = CARPETA_RESULTADOS / 'metricas_resumen.csv'
    df_resultados.to_csv(ruta_csv, index=False)
    print(f"\nTabla resumen guardada en: {ruta_csv}")

## 8. Visualización de resultados

In [ ]:
import matplotlib.pyplot as plt

# Volver a cargar por si se ejecuta esta celda sola
resultados_viz = []
for json_path in sorted(CARPETA_RESULTADOS.glob('metricas_B*.json')):
    with open(json_path) as f:
        resultados_viz.append(json.load(f))

if not resultados_viz:
    print("No hay resultados para graficar.")
else:
    df_viz = pd.DataFrame([
        {
            'Experimento': r['experimento'],
            'Fold': r['fold'],
            'Accuracy': r['accuracy'],
            'F1': r['f1'],
            'AUC': r['auc'],
        }
        for r in resultados_viz
    ])
    
    # Graficar accuracy por experimento (boxplot)
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    metricas_a_graficar = ['Accuracy', 'F1', 'AUC']
    
    for ax, metrica in zip(axes, metricas_a_graficar):
        # Agrupar por experimento
        grupos = [df_viz[df_viz['Experimento'] == exp][metrica].values
                  for exp in ['B1', 'B2', 'B3', 'B4']]
        
        ax.boxplot(grupos, labels=['B1\nMel', 'B2\nMFCC', 'B3\nFFT', 'B4\nCQT'])
        ax.set_title(metrica)
        ax.set_ylabel(metrica)
        ax.set_ylim(0, 1)
        ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='azar')
        ax.grid(axis='y', alpha=0.3)
    
    axes[0].legend()
    fig.suptitle('xnorC — Resultados por representación de audio (k-hive-fold)', fontsize=13)
    plt.tight_layout()
    plt.savefig(CARPETA_RESULTADOS / 'grafico_resultados.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Gráfico guardado en: {CARPETA_RESULTADOS / 'grafico_resultados.png'}")

In [ ]:
# Graficar la curva de aprendizaje de un fold específico
# (útil para ver si hubo overfitting o si el early stopping funcionó bien)

EXPERIMENTO_VIZ = 'B1'  # cambiá esto para ver otro experimento
FOLD_VIZ = 0            # cambiá esto para ver otro fold

ruta_modelo_viz = CARPETA_MODELOS / f'xnorC_{EXPERIMENTO_VIZ}_fold{FOLD_VIZ}.pt'

if not ruta_modelo_viz.exists():
    print(f"No se encontró el modelo: {ruta_modelo_viz}")
else:
    checkpoint = torch.load(ruta_modelo_viz, map_location='cpu')
    historial = checkpoint['historial']
    
    epocas = [h['epoca'] for h in historial]
    perdida_train = [h['perdida_train'] for h in historial]
    perdida_test  = [h['perdida_test']  for h in historial]
    accuracy_test = [h['accuracy']      for h in historial]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Pérdida
    ax1.plot(epocas, perdida_train, label='train', color='steelblue')
    ax1.plot(epocas, perdida_test,  label='test',  color='coral')
    ax1.set_title(f'Pérdida — {EXPERIMENTO_VIZ} fold {FOLD_VIZ}')
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Cross-entropy loss')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # Accuracy
    ax2.plot(epocas, accuracy_test, color='coral')
    ax2.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='azar')
    ax2.set_title(f'Accuracy en test — {EXPERIMENTO_VIZ} fold {FOLD_VIZ}')
    ax2.set_xlabel('Época')
    ax2.set_ylabel('Accuracy')
    ax2.set_ylim(0, 1)
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(CARPETA_RESULTADOS / f'curva_{EXPERIMENTO_VIZ}_fold{FOLD_VIZ}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Curva de entrenamiento: {len(epocas)} épocas efectivas")

## Notas sobre los resultados

### Qué esperar

xnorC es un clasificador de pares, no de colmenas individuales. El accuracy reportado
acá mide cuántos pares clasifica bien (IGUAL/DIFERENTE).

Un modelo que adivina al azar debería tener accuracy ≈ 0.5 (porque los pares están
balanceados al 50/50 por la regla R3).

### Comparación entre representaciones

El OE2B-04 mostró que B4 (CQT) tiene mejor separación QR/QL en promedio.
Es esperable que B4 tenga los mejores resultados, aunque no garantizado:
xnorC puede aprender a separar clases de maneras que el promedio temporal no captura.

### Siguiente paso

El OE2B-06 evalúa los modelos entrenados acá usando cross-hive-fold:
mide qué tan bien generaliza xnorC a colmenas que nunca vio durante el entrenamiento.